In [2]:
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple

# Sklearn for scaling
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# SciPy for acquisition functions and optimisation
from scipy.stats import norm
from scipy.optimize import minimize

# PyTorch for deep ensemble surrogate
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 0. Global settings
# ============================================================

RANDOM_STATE = 123
N_ENSEMBLE = 15          # number of neural nets in the ensemble
N_EPOCHS = 1500          # training epochs per ensemble member
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
DEVICE = torch.device("cpu")  # change to "cuda" if you have a GPU and want to use it


# ============================================================
# 1. Data (as provided)
# ============================================================

X_raw = np.array([
    [6.04994453e-01, 2.92215020e-01, 9.08452748e-01, 3.55506242e-01,
     2.01668719e-01, 5.75338005e-01, 3.10310951e-01, 7.34281377e-01],
    [1.78006959e-01, 5.66222654e-01, 9.94861845e-01, 2.10325006e-01,
     3.20152657e-01, 7.07908792e-01, 6.35384489e-01, 1.07131627e-01],
    [9.07697668e-03, 8.11626153e-01, 5.20520360e-01, 7.56866752e-02,
     2.65111825e-01, 9.16516894e-02, 5.92415145e-01, 3.67320262e-01],
    [5.06028164e-01, 6.53730123e-01, 3.63410779e-01, 1.77981049e-01,
     9.37283044e-02, 1.97425331e-01, 7.55826900e-01, 2.92472339e-01],
    [3.59909264e-01, 2.49075679e-01, 4.95997170e-01, 7.09214981e-01,
     1.14987195e-01, 2.89206921e-01, 5.57295151e-01, 5.93881726e-01],
    [7.78818344e-01, 3.41949948e-03, 3.37983130e-01, 5.19527778e-01,
     8.20906993e-01, 5.37246689e-01, 5.51347098e-01, 6.60032086e-01],
    [9.08649322e-01, 6.22496998e-02, 2.38259546e-01, 7.66603545e-01,
     1.32335962e-01, 9.90243814e-01, 6.88067822e-01, 7.42495941e-01],
    [5.86371444e-01, 8.80735726e-01, 7.45020752e-01, 5.46034849e-01,
     9.64887799e-03, 7.48991763e-01, 2.30907070e-01, 9.79156228e-02],
    [7.61137326e-01, 8.54672390e-01, 3.82124331e-01, 3.37351983e-01,
     6.89708320e-01, 3.09853052e-01, 6.31379683e-01, 4.19560695e-02],
    [9.84933202e-01, 6.99506258e-01, 9.98885497e-01, 1.80148456e-01,
     5.80143147e-01, 2.31087191e-01, 4.90826936e-01, 3.13682720e-01],
    [1.12071314e-01, 4.37735663e-01, 5.96598785e-01, 5.92775633e-01,
     2.26981770e-01, 4.10104519e-01, 9.21237577e-01, 6.74752759e-01],
    [7.91887508e-01, 5.76191336e-01, 6.94528359e-01, 2.83423782e-01,
     1.36755461e-01, 2.79161861e-01, 8.42767264e-01, 6.25327922e-01],
    [1.43550296e-01, 9.37414515e-01, 2.32324818e-01, 9.04348540e-03,
     4.14578930e-01, 4.09325169e-01, 5.53778522e-01, 2.05840800e-01],
    [7.69916548e-01, 4.58759088e-01, 5.59000445e-01, 6.94604441e-01,
     5.03199022e-01, 7.28346383e-01, 7.84253534e-01, 6.63131087e-01],
    [5.64474111e-02, 6.59555534e-02, 2.29286780e-02, 3.87864724e-02,
     4.03935441e-01, 8.01055329e-01, 4.88307007e-01, 8.93084977e-01],
    [8.62437445e-01, 4.82733822e-01, 2.81869398e-01, 5.44102227e-01,
     8.87490260e-01, 3.82654693e-01, 6.01901993e-01, 4.76461690e-01],
    [3.51511904e-01, 5.90064942e-01, 9.09436304e-01, 6.78408354e-01,
     2.12825656e-01, 8.84603803e-02, 4.10152995e-01, 1.95724292e-01],
    [7.35903638e-01, 3.46118895e-02, 7.28030269e-01, 1.47426522e-01,
     2.95743139e-01, 4.45117308e-01, 9.75179686e-01, 3.74339784e-01],
    [6.80293974e-01, 2.55104646e-01, 8.62187985e-01, 1.34395821e-01,
     3.26329200e-01, 2.87906871e-01, 4.35010484e-01, 3.64200126e-01],
    [4.43292532e-02, 1.35814872e-02, 2.58198240e-01, 5.77644163e-01,
     5.12799230e-02, 1.58563071e-01, 5.91030124e-01, 7.79529335e-02],
    [7.78345480e-01, 7.51145652e-01, 3.14142208e-01, 9.02985775e-01,
     3.35381656e-01, 3.86322669e-01, 7.48972486e-01, 9.88755104e-01],
    [8.98887111e-01, 5.23641705e-01, 8.76783255e-01, 2.18696449e-01,
     9.00260894e-01, 2.82766245e-01, 9.11077910e-01, 4.72398218e-01],
    [1.45120286e-01, 1.19327540e-01, 4.20888224e-01, 3.87608607e-01,
     1.55422833e-01, 8.75171626e-01, 5.10559672e-01, 7.28610579e-01],
    [3.38954419e-01, 5.66932018e-01, 3.76751098e-01, 9.89157290e-02,
     6.59451687e-01, 2.45548090e-01, 7.62482784e-01, 7.32153467e-01],
    [1.76150018e-01, 2.93961428e-01, 9.75679966e-01, 7.93936306e-01,
     9.23400762e-01, 3.08422938e-02, 8.03254524e-01, 5.95897583e-01],
    [2.89466302e-02, 2.82790578e-02, 4.81371555e-01, 6.13174600e-01,
     6.72660448e-01, 2.21134069e-02, 6.01483302e-01, 5.24885053e-01],
    [1.92639868e-01, 6.30677279e-01, 4.16795837e-01, 4.90529289e-01,
     7.96086023e-01, 6.54567065e-01, 2.76241193e-01, 2.95517586e-01],
    [9.43185017e-01, 2.18850618e-01, 7.21184081e-01, 4.24597072e-01,
     9.86902000e-01, 5.35182984e-01, 7.14743177e-01, 9.60093720e-01],
    [5.32721401e-01, 8.33692597e-01, 7.13990037e-02, 1.16811483e-01,
     7.30693110e-01, 9.37375591e-01, 8.66507981e-01, 1.27901999e-01],
    [4.47095841e-01, 8.43952527e-01, 7.29546115e-01, 6.39151378e-01,
     4.09287137e-01, 1.32645694e-01, 3.59088762e-02, 4.46838470e-01],
    [3.82224965e-01, 5.57135837e-01, 8.53101634e-01, 3.33795692e-01,
     2.65721272e-01, 4.80872916e-01, 2.37647062e-01, 7.68631956e-01],
    [5.32819530e-01, 8.62308484e-01, 5.38267119e-01, 4.94429349e-02,
     7.19701189e-01, 9.06705899e-01, 1.08230943e-01, 5.25347913e-01],
    [3.94865187e-01, 3.31801666e-01, 7.40754301e-01, 6.97861725e-01,
     7.37404439e-01, 7.83776810e-01, 2.54495461e-01, 8.71145510e-01],
    [9.85945390e-01, 8.73053629e-01, 7.03926194e-02, 5.35872927e-02,
     7.34152958e-01, 5.20258522e-01, 8.11040045e-01, 1.03360365e-01],
    [9.64573386e-01, 9.73979787e-01, 6.63753351e-01, 6.62215992e-01,
     6.73121672e-01, 9.05237624e-01, 4.58874624e-01, 5.60917502e-01],
    [4.72070709e-01, 1.68202645e-01, 8.64275662e-02, 4.52655513e-01,
     4.80619220e-01, 6.22439489e-01, 9.28974462e-01, 1.12536267e-01],
    [8.56006953e-01, 6.38893704e-01, 3.26192022e-01, 6.68503115e-01,
     2.40298369e-01, 2.10298890e-01, 1.67546362e-01, 9.63589863e-01],
    [8.10031736e-01, 6.35046041e-01, 2.69547579e-01, 8.69605338e-01,
     6.61921590e-01, 2.52258727e-01, 7.65670033e-01, 8.90548667e-01],
    [7.96252524e-01, 7.03653252e-03, 3.55697380e-01, 4.87566053e-01,
     7.40519615e-01, 7.06650103e-01, 9.92914495e-01, 3.81734367e-01],
    [4.81245331e-01, 1.02460721e-01, 2.19485939e-01, 6.77322369e-01,
     2.47509187e-01, 2.44340858e-01, 1.63824527e-01, 7.15961640e-01],
    [1.08594500e+00, 1.07397900e+00, 1.09888500e+00, 1.00298500e+00,
     1.08690100e+00, 1.09024300e+00, 1.09291400e+00, 1.09291500e+00],
    [1.00000000e-06, 2.05513000e-01, 1.00000000e-06, 8.42570000e-02,
     8.08305000e-01, 3.55768000e-01, 1.00000000e-06, 4.52510000e-01],
    [2.55540000e-02, 3.41676000e-01, 2.25910000e-02, 3.51417000e-01,
     5.40332000e-01, 5.67680000e-01, 2.14489000e-01, 5.95807000e-01],
    [1.57856000e-01, 1.76691000e-01, 2.44910000e-02, 4.77665000e-01,
     9.06327000e-01, 7.15038000e-01, 2.23940000e-02, 2.50892000e-01],
])

y_raw = np.array([
    7.3987211, 7.00522736, 8.45948162, 8.28400781, 8.60611679,
    8.54174792, 7.32743458, 7.29987205, 7.95787474, 5.59219339,
    7.85454099, 6.79198578, 8.97655402, 7.3790829,  9.598482,
    8.15998319, 7.13162397, 6.76796253, 7.43374407, 9.01307515,
    7.31089382, 5.84106731, 9.14163949, 8.81755844, 6.45194313,
    8.83074505, 9.34427428, 6.88784639, 8.04221254, 7.69236805,
    7.92375877, 8.42175924, 8.2780624,  7.11345716, 6.40258841,
    8.47293632, 7.97768459, 7.46087219, 7.43659353, 9.18300525,
    1.64985965, 9.81888546, 9.83828107, 9.72466268
])

assert X_raw.shape[0] == y_raw.shape[0], "X and y sizes must match"

# ============================================================
# 2. PyTorch Deep Ensemble Surrogate
# ============================================================

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


def build_ensemble_torch(n_members: int, input_dim: int) -> List[MLPRegressorTorch]:
    torch.manual_seed(RANDOM_STATE)
    ensemble = []
    for _ in range(n_members):
        model = MLPRegressorTorch(input_dim).to(DEVICE)
        ensemble.append(model)
    return ensemble


def fit_ensemble_torch(ensemble: List[MLPRegressorTorch],
                       X_scaled: np.ndarray,
                       y_scaled: np.ndarray,
                       n_epochs: int = N_EPOCHS,
                       batch_size: int = BATCH_SIZE,
                       lr: float = LEARNING_RATE) -> None:
    """
    Train each ensemble member on a bootstrap sample of the data.
    """
    rng = np.random.RandomState(RANDOM_STATE + 1)
    X = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)
    y = torch.from_numpy(y_scaled.astype(np.float32)).view(-1, 1).to(DEVICE)

    n_samples = X.shape[0]

    for i, model in enumerate(ensemble):
        # Bootstrap indices with replacement
        idx = rng.randint(0, n_samples, size=n_samples)
        X_boot = X[idx]
        y_boot = y[idx]

        dataset = TensorDataset(X_boot, y_boot)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.MSELoss()

        for epoch in range(n_epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()


def ensemble_predict_torch(ensemble: List[MLPRegressorTorch],
                           X_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Predict mean and standard deviation across the ensemble for a batch of points.
    X_scaled: (N, D) numpy (already scaled with MinMaxScaler).
    """
    X_t = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)

    preds_list = []
    for model in ensemble:
        with torch.no_grad():
            preds = model(X_t).cpu().numpy().ravel()
        preds_list.append(preds)

    preds = np.stack(preds_list, axis=0)  # (n_members, N)
    mu = preds.mean(axis=0)
    sigma = preds.std(axis=0, ddof=1) + 1e-9
    return mu, sigma


# ============================================================
# 3. Acquisition functions (vectorised, numpy)
# ============================================================

def probability_of_improvement(mu: np.ndarray,
                               sigma: np.ndarray,
                               best_y: float,
                               xi: float = 0.01) -> np.ndarray:
    """
    Probability that f(x) exceeds current best_y by at least xi.
    """
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - best_y - xi) / sigma_safe
    return norm.cdf(z)


def expected_improvement(mu: np.ndarray,
                         sigma: np.ndarray,
                         best_y: float,
                         xi: float = 0.01) -> np.ndarray:
    """
    Classical Bayesian optimisation Expected Improvement.
    """
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - best_y - xi) / sigma_safe
    ei = (mu - best_y - xi) * norm.cdf(z) + sigma_safe * norm.pdf(z)
    ei[sigma_safe < 1e-12] = 0.0
    return ei


# ============================================================
# 4. EI and gradient for a single point (PyTorch autograd)
# ============================================================

def ei_and_grad_single(
    x_raw_1d: np.ndarray,
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    best_y: float,
    xi: float = 0.01,
) -> Tuple[float, np.ndarray]:
    """
    Compute EI(x) and its gradient wrt x_raw for a SINGLE point.
    Uses PyTorch autograd on the deep ensemble surrogate.
    """
    # Ensure shape (1, D)
    x_raw = x_raw_1d.reshape(1, -1)

    # Scale input to model space
    x_scaled = x_scaler.transform(x_raw)  # (1, D)
    x_t = torch.tensor(x_scaled.astype(np.float32), requires_grad=True, device=DEVICE)

    preds = []
    for model in ensemble:
        y_pred = model(x_t)  # (1, 1)
        preds.append(y_pred)

    # (n_members, 1, 1) -> (n_members,)
    Y = torch.stack(preds, dim=0).squeeze(-1).squeeze(-1)

    mu_scaled = Y.mean()
    sigma_scaled = Y.std(unbiased=True) + 1e-9

    # Back to original y scale
    scale_y = float(y_scaler.scale_[0])
    mean_y = float(y_scaler.mean_[0])
    mu_y = mu_scaled * scale_y + mean_y
    sigma_y = sigma_scaled * scale_y
    sigma_y_safe = torch.clamp(sigma_y, min=1e-8)

    normal = torch.distributions.Normal(0.0, 1.0)
    z = (mu_y - best_y - xi) / sigma_y_safe
    cdf = normal.cdf(z)
    pdf = torch.exp(normal.log_prob(z))

    ei = (mu_y - best_y - xi) * cdf + sigma_y_safe * pdf

    # Backprop EI wrt x_scaled
    ei.backward()
    grad_x_scaled = x_t.grad.detach().cpu().numpy().reshape(-1)  # dEI/dx_scaled

    # Chain rule: x_scaled = scale_ * x_raw + min_
    # So dEI/dx_raw = dEI/dx_scaled * scale_
    grad_x_raw = grad_x_scaled * x_scaler.scale_.astype(np.float32)

    ei_value = float(ei.detach().cpu().item())
    return ei_value, grad_x_raw


# ============================================================
# 5. BO result dataclass
# ============================================================

@dataclass
class BOResult:
    x_next: np.ndarray
    mu_next: float
    sigma_next: float
    pi_next: float
    ei_next: float
    x_best_observed: np.ndarray
    y_best_observed: float
    reasoning: str


# ============================================================
# 6. Propose next query point with random search + SciPy refinement
# ============================================================

def propose_next_point(
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    X_raw: np.ndarray,
    y_raw: np.ndarray,
    n_candidates: int = 50_000,
    n_restarts: int = 10,
    xi: float = 0.01,
    random_seed: int = 1234,
) -> BOResult:
    """
    1) Sample candidate points uniformly in [0,1]^8 and evaluate EI.
    2) Take top-n_restarts candidates and run a gradient-based SciPy
       optimiser (L-BFGS-B) to refine EI.
    3) Return the best EI point found.
    """

    rng = np.random.RandomState(random_seed)

    # Current best from observed data (original y scale)
    best_idx = np.argmax(y_raw)
    x_best_obs = X_raw[best_idx]
    y_best_obs = float(y_raw[best_idx])

    # Generate random candidates in [0,1]^8
    dim = X_raw.shape[1]
    X_cand_raw = rng.rand(n_candidates, dim)  # search domain is [0,1]^8

    # Scale to model space
    X_cand_scaled = x_scaler.transform(X_cand_raw)

    # Vectorised ensemble predictions
    mu_scaled, sigma_scaled = ensemble_predict_torch(ensemble, X_cand_scaled)

    # Back to original y scale
    mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
    sigma = sigma_scaled * y_scaler.scale_[0]

    # Acquisition values
    ei = expected_improvement(mu, sigma, y_best_obs, xi=xi)
    pi = probability_of_improvement(mu, sigma, y_best_obs, xi=xi)

    # Random-search best candidate (baseline)
    best_cand_idx = np.argmax(ei)
    x_next_raw = X_cand_raw[best_cand_idx]
    mu_next = float(mu[best_cand_idx])
    sigma_next = float(sigma[best_cand_idx])
    pi_next = float(pi[best_cand_idx])
    ei_next = float(ei[best_cand_idx])

    # --------------------------------------------------------
    # Gradient-based refinement of EI using SciPy
    # --------------------------------------------------------

    def make_objective():
        def obj(x: np.ndarray) -> Tuple[float, np.ndarray]:
            # EI(x) and grad_EI(x)
            ei_val, grad = ei_and_grad_single(
                x_raw_1d=x,
                ensemble=ensemble,
                x_scaler=x_scaler,
                y_scaler=y_scaler,
                best_y=y_best_obs,
                xi=xi,
            )
            # SciPy minimises, so we minimise -EI
            f = -ei_val
            g = -grad  # gradient of -EI
            return f, g
        return obj

    objective = make_objective()
    bounds = [(0.0, 1.0)] * dim

    # Choose top-n_restarts seeds by EI for refinement
    top_indices = np.argsort(-ei)[:n_restarts]
    best_f = -ei_next  # current best objective value = -EI
    best_x = x_next_raw.copy()

    for idx in top_indices:
        x0 = X_cand_raw[idx]

        res = minimize(
            fun=lambda x: objective(x)[0],
            x0=x0,
            method="L-BFGS-B",
            jac=lambda x: objective(x)[1],
            bounds=bounds,
            options={"maxiter": 100},
        )

        if not res.success:
            continue

        f_val = res.fun  # this is -EI
        if f_val < best_f:
            best_f = f_val
            best_x = res.x

    # Evaluate final refined point
    x_refined_raw = best_x.reshape(1, -1)
    x_refined_scaled = x_scaler.transform(x_refined_raw)

    mu_scaled_ref, sigma_scaled_ref = ensemble_predict_torch(ensemble, x_refined_scaled)
    mu_ref = float(y_scaler.inverse_transform(mu_scaled_ref.reshape(-1, 1))[0, 0])
    sigma_ref = float(sigma_scaled_ref[0] * y_scaler.scale_[0])

    pi_ref = float(
        probability_of_improvement(
            np.array([mu_ref]), np.array([sigma_ref]), y_best_obs, xi=xi
        )[0]
    )
    ei_ref = float(
        expected_improvement(
            np.array([mu_ref]), np.array([sigma_ref]), y_best_obs, xi=xi
        )[0]
    )

    # Build reasoning text
    reasoning_lines = [
        f"Observed data maximum: y = {y_best_obs:.4f} at x = {x_best_obs}.",
        f"Random search over {n_candidates} points in [0,1]^8 found an EI-best candidate",
        f"  with EI ≈ {ei_next:.4f} at x ≈ {x_next_raw}.",
        "On top of that, a gradient-based L-BFGS-B optimisation of EI was run",
        f"  starting from the top {n_restarts} random EI seeds.",
        f"The best refined point after optimisation is x_next ≈ {x_refined_raw.ravel()},",
        f"  with predicted mean μ ≈ {mu_ref:.4f}, uncertainty σ ≈ {sigma_ref:.4f},",
        f"  probability of improvement PI ≈ {pi_ref:.3f}, and EI ≈ {ei_ref:.4f}.",
        "This refined candidate achieves a higher EI than the pure random-search best,",
        "so it is our recommended next query point to most likely improve the current local maximum.",
    ]
    reasoning = "\n".join(reasoning_lines)

    return BOResult(
        x_next=x_refined_raw.ravel(),
        mu_next=mu_ref,
        sigma_next=sigma_ref,
        pi_next=pi_ref,
        ei_next=ei_ref,
        x_best_observed=x_best_obs,
        y_best_observed=y_best_obs,
        reasoning=reasoning,
    )


# ============================================================
# 7. Main script
# ============================================================

def main():
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    print("=== 8D Black-Box Optimisation with PyTorch Deep NN Ensemble ===")

    # ---- Scale inputs and outputs ----
    # Inputs: scaled to [0,1] based on current data.
    # Search bounds for next point are explicitly [0,1]^8.
    x_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X_scaled = x_scaler.fit_transform(X_raw)

    # Outputs: standardised for stable NN training
    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # ---- Build & train ensemble surrogate ----
    print("Training PyTorch deep neural-network ensemble surrogate...")
    input_dim = X_raw.shape[1]
    ensemble = build_ensemble_torch(N_ENSEMBLE, input_dim)
    fit_ensemble_torch(ensemble, X_scaled, y_scaled)
    print("Training complete.\n")

    # ---- Propose next query point (random search + gradient-based EI refinement) ----
    result = propose_next_point(
        ensemble=ensemble,
        x_scaler=x_scaler,
        y_scaler=y_scaler,
        X_raw=X_raw,
        y_raw=y_raw,
        n_candidates=50_000,
        n_restarts=10,
        xi=0.01,
        random_seed=RANDOM_STATE + 5,
    )

    # ---- Report results ----
    print("=== CURRENT BEST (observed data) ===")
    print(f"x_best (observed) = {result.x_best_observed}")
    print(f"y_best (observed) = {result.y_best_observed:.6f}\n")

    print("=== PROPOSED NEXT QUERY POINT (within [0,1]^8) ===")
    print(f"x_next         = {result.x_next}")
    print(f"Predicted μ    = {result.mu_next:.6f}")
    print(f"Predicted σ    = {result.sigma_next:.6f}")
    print(f"Prob. of Improvement (PI) ≈ {result.pi_next:.3f}")
    print(f"Expected Improvement (EI) = {result.ei_next:.6f}\n")

    print("=== REASONING ===")
    print(result.reasoning)
    print("\nDone.")


if __name__ == "__main__":
    main()


=== 8D Black-Box Optimisation with PyTorch Deep NN Ensemble ===
Training PyTorch deep neural-network ensemble surrogate...
Training complete.

=== CURRENT BEST (observed data) ===
x_best (observed) = [0.025554 0.341676 0.022591 0.351417 0.540332 0.56768  0.214489 0.595807]
y_best (observed) = 9.838281

=== PROPOSED NEXT QUERY POINT (within [0,1]^8) ===
x_next         = [0.         0.17198568 0.         0.28222258 1.         1.
 0.         1.        ]
Predicted μ    = 10.099997
Predicted σ    = 0.180177
Prob. of Improvement (PI) ≈ 0.919
Expected Improvement (EI) = 0.258365

=== REASONING ===
Observed data maximum: y = 9.8383 at x = [0.025554 0.341676 0.022591 0.351417 0.540332 0.56768  0.214489 0.595807].
Random search over 50000 points in [0,1]^8 found an EI-best candidate
  with EI ≈ 0.0796 at x ≈ [0.026436   0.2221     0.01956195 0.08974339 0.98134846 0.61150394
 0.06684845 0.6535583 ].
On top of that, a gradient-based L-BFGS-B optimisation of EI was run
  starting from the top 10 ra